<a href="https://colab.research.google.com/github/aashnikatari/eeg_classification/blob/eeg_sandbox/EEG_Data_Analysis_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# TDBRAIN EEG Analysis for OCD vs Anxiety Disorder - Starter Notebook
# =============================================================================
# This notebook provides a complete pipeline for:
# 1. Loading and preprocessing TDBRAIN EEG data
# 2. Downsampling to match cross-dataset analysis (256 Hz)
# 3. Feature extraction for ML models
# 4. Basic ML model training (SVM, Random Forest, LSTM)
# =============================================================================

# Install required packages
!pip install mne pandas numpy scikit-learn matplotlib seaborn -q
!pip install tensorflow -q  # For LSTM models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 50.9 MB/s eta 0:00:00


In [2]:
# =============================================================================
# Cell 2: Import Libraries and Configuration
# =============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from mne.io import read_raw_brainvision
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Deep Learning (LSTM)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Configuration
TARGET_SFREQ = 256  # Target sampling frequency (to match Mendeley dataset)
EPOCH_DURATION = 10  # Seconds per epoch
FREQ_BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha1': (8, 10),
    'alpha2': (10, 13),
    'beta1': (13, 20),
    'beta2': (20, 30)
}

# Common channels between TDBRAIN (10-10) and Mendeley (10-20)
COMMON_CHANNELS = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
                   'T7', 'C3', 'Cz', 'C4', 'T8',
                   'P7', 'P3', 'Pz', 'P4', 'P8',
                   'O1', 'O2']

print("Libraries loaded successfully!")
print(f"MNE version: {mne.__version__}")
print(f"TensorFlow version: {tf.__version__}")

Libraries loaded successfully!
MNE version: 1.11.0
TensorFlow version: 2.19.0


## Create `eeg_loader.py` Module

This cell writes the EEG loading and preprocessing functions into a separate Python file. This improves modularity and organization.

In [14]:
%%writefile eeg_loader.py

import os
import mne
from mne.io import read_raw_brainvision
import warnings
warnings.filterwarnings('ignore')

# Configuration (assuming these are globally available or passed)
TARGET_SFREQ = 256
COMMON_CHANNELS = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
                   'T7', 'C3', 'Cz', 'C4', 'T8',
                   'P7', 'P3', 'Pz', 'P4', 'P8',
                   'O1', 'O2']

def load_and_preprocess_eeg(vhdr_file, target_sfreq=TARGET_SFREQ, common_channels=COMMON_CHANNELS):
    """
    Load a BrainVision EEG file, select common channels, and downsample.

    Parameters:
    -----------
    vhdr_file : str
        Path to the .vhdr file
    target_sfreq : int
        Target sampling frequency (default: 256 Hz to match Mendeley)
    common_channels : list
        List of channel names to keep (for cross-dataset compatibility)

    Returns:
    --------
    raw : mne.io.Raw
        Preprocessed raw EEG data
    """
    try:
        # Load raw EEG data
        raw = read_raw_brainvision(vhdr_file, preload=True, verbose=False)
        original_sfreq = raw.info['sfreq']

        # Select only common channels if specified
        if common_channels:
            available_channels = [ch for ch in common_channels if ch in raw.ch_names]
            if len(available_channels) < len(common_channels):
                missing = set(common_channels) - set(available_channels)
                print(f"  Warning: Missing channels: {missing}")
            raw.pick_channels(available_channels)

        # Downsample if needed
        if raw.info['sfreq'] != target_sfreq:
            raw.resample(target_sfreq, verbose=False)
            print(f"  Downsampled: {original_sfreq} Hz -> {target_sfreq} Hz")

        # Apply basic filtering (0.5-45 Hz bandpass)
        raw.filter(0.5, 45, verbose=False)

        return raw

    except Exception as e:
        print(f"  Error loading {vhdr_file}: {e}")
        return None

def find_eeg_files(subject_path, task='restEC'):
    """
    Find EEG files for a given subject and task.

    Parameters:
    -----------
    subject_path : str
        Path to subject folder
    task : str
        Task name (restEC for eyes-closed, restEO for eyes-open)

    Returns:
    --------
    vhdr_file : str or None
        Path to .vhdr file if found
    """
    eeg_path = os.path.join(subject_path, 'ses-1/eeg')
    if not os.path.exists(eeg_path):
        return None

    for f in os.listdir(eeg_path):
        if f.endswith('.vhdr') and task in f:
            return os.path.join(eeg_path, f)
    return None

print("eeg_loader.py created successfully!")

Overwriting eeg_loader.py


In [15]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


To confirm the `eeg_loader.py` file exists in the Colab runtime's local file system and view its content, execute the following command:

In [ ]:
!cat /content/eeg_loader.py

If you wish to save this `eeg_loader.py` file to your Google Drive (e.g., in your `OUTPUT_PATH` folder), you can copy it using the command below:

In [ ]:
# Copy eeg_loader.py to your Google Drive output folder
!cp /content/eeg_loader.py "{OUTPUT_PATH}/eeg_loader.py"
print(f"eeg_loader.py copied to {OUTPUT_PATH}/eeg_loader.py")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Import EEG Loading Functions from Module

Now, we import the `load_and_preprocess_eeg` and `find_eeg_files` functions from the `eeg_loader` module we just created. This allows us to use these functions throughout the notebook without having them directly defined here.

In [4]:
from eeg_loader import load_and_preprocess_eeg, find_eeg_files

print("EEG loading functions imported from eeg_loader.py!")

eeg_loader.py created successfully!
EEG loading functions imported from eeg_loader.py!


In [5]:
# =============================================================================
# Cell 3: Mount Google Drive & Setup Data Paths
# =============================================================================
# Upload your TDBRAIN subset to Google Drive before running this

from google.colab import drive
drive.mount('/content/drive')

# Set your data paths (modify these based on your folder structure)
TDBRAIN_PATH = '/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/'  # Modify this path
OUTPUT_PATH = '/content/drive/MyDrive/01_Research/01_Neuro_EEG/EEG_Analysis_Output/'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Expected folder structure:
# TDBRAIN_subset/
#   ├── participants.tsv        # Participant metadata with diagnosis
#   ├── sub-001/
#   │   └── eeg/
#   │       ├── sub-001_task-restEC_eeg.vhdr
#   │       ├── sub-001_task-restEC_eeg.vmrk
#   │       └── sub-001_task-restEC_eeg.eeg
#   ├── sub-002/
#   │   └── ...

print(f"TDBRAIN path: {TDBRAIN_PATH}")
print(f"Output path: {OUTPUT_PATH}")

Mounted at /content/drive
TDBRAIN path: /content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/
Output path: /content/drive/MyDrive/EEG_Analysis_Output/


In [11]:
# =============================================================================
# Cell 4: Load Participant Metadata and Filter OCD Subjects
# =============================================================================

def load_participants_metadata(tdbrain_path):
    """
    Load TDBRAIN participants.tsv file and filter for OCD subjects.
    """
    participants_file = os.path.join(tdbrain_path, 'participants.tsv')

    if not os.path.exists(participants_file):
        print(f"Warning: {participants_file} not found!")
        print("Creating sample metadata structure...")
        # Create a dummy DataFrame if the file is not found
        # This will prevent NameError in subsequent cells for demonstration purposes
        sample_data = {'participant_id': [f'sub-{i:03d}' for i in range(1, 31)],
                       'diagnosis': ['ADHD', 'SMC'] * 15} # Example diagnoses
        return pd.DataFrame(sample_data)

    # Load the TSV file
    df = pd.read_csv(participants_file, sep='\t')
    print(f"Total participants: {len(df)}")
    print(f"\nColumns: {df.columns.tolist()}")

    # Display diagnosis distribution
    if 'diagnosis' in df.columns:
        print(f"\nDiagnosis distribution:")
        print(df['diagnosis'].value_counts())
    elif 'indication' in df.columns:
        print(f"\nIndication distribution:")
        print(df['indication'].value_counts())

    return df

# Load metadata
participants_df = load_participants_metadata(TDBRAIN_PATH)

# Filter OCD subjects (adjust column name based on your metadata)
if participants_df is not None:
    # Common column names in TDBRAIN: 'diagnosis', 'indication', 'group'
    diag_col = 'diagnosis' if 'diagnosis' in participants_df.columns else 'indication'
    # Ensure the column exists before filtering
    if diag_col in participants_df.columns:
        ocd_subjects = participants_df[participants_df[diag_col].str.contains(r'ADHD|SMC', case=False, na=False)]
    else:
        print(f"Warning: Neither 'diagnosis' nor 'indication' column found. Creating empty ocd_subjects DataFrame.")
        ocd_subjects = pd.DataFrame(columns=['participant_id', 'diagnosis'])
    print(f"\nOCD subjects found: {len(ocd_subjects)}")
    # Only try to print if there are columns to print
    if not ocd_subjects.empty and not ocd_subjects.columns.intersection(['participant_id', diag_col]).empty:
        print(ocd_subjects[['participant_id', diag_col]].head(10))
else:
    # If participants_df is None (should not happen with the dummy DataFrame logic now),
    # create an empty ocd_subjects DataFrame to prevent NameError
    ocd_subjects = pd.DataFrame(columns=['participant_id', 'diagnosis'])
    print("Warning: participants_df was None. Created empty ocd_subjects DataFrame.")


Creating sample metadata structure...

OCD subjects found: 30
  participant_id diagnosis
0        sub-001      ADHD
1        sub-002       SMC
2        sub-003      ADHD
3        sub-004       SMC
4        sub-005      ADHD
5        sub-006       SMC
6        sub-007      ADHD
7        sub-008       SMC
8        sub-009      ADHD
9        sub-010       SMC


In [10]:
# =============================================================================
# Cell 5: EEG Loading and Downsampling Functions
# =============================================================================

# These functions are now loaded from eeg_loader.py
# def load_and_preprocess_eeg(vhdr_file, target_sfreq=256, common_channels=None):
#     """
#     Load a BrainVision EEG file, select common channels, and downsample.

#     Parameters:
#     -----------
#     vhdr_file : str
#         Path to the .vhdr file
#     target_sfreq : int
#         Target sampling frequency (default: 256 Hz to match Mendeley)
#     common_channels : list
#         List of channel names to keep (for cross-dataset compatibility)

#     Returns:
#     --------
#     raw : mne.io.Raw
#         Preprocessed raw EEG data
#     """
#     try:
#         # Load raw EEG data
#         raw = read_raw_brainvision(vhdr_file, preload=True, verbose=False)
#         original_sfreq = raw.info['sfreq']

#         # Select only common channels if specified
#         if common_channels:
#             available_channels = [ch for ch in common_channels if ch in raw.ch_names]
#             if len(available_channels) < len(common_channels):
#                 missing = set(common_channels) - set(available_channels)
#                 print(f"  Warning: Missing channels: {missing}")
#             raw.pick_channels(available_channels)

#         # Downsample if needed
#         if raw.info['sfreq'] != target_sfreq:
#             raw.resample(target_sfreq, verbose=False)
#             print(f"  Downsampled: {original_sfreq} Hz -> {target_sfreq} Hz")

#         # Apply basic filtering (0.5-45 Hz bandpass)
#         raw.filter(0.5, 45, verbose=False)

#         return raw

#     except Exception as e:
#         print(f"  Error loading {vhdr_file}: {e}")
#         return None

# def find_eeg_files(subject_path, task='restEC'):
#     """
#     Find EEG files for a given subject and task.

#     Parameters:
#     -----------
#     subject_path : str
#         Path to subject folder
#     task : str
#         Task name (restEC for eyes-closed, restEO for eyes-open)

#     Returns:
#     --------
#     vhdr_file : str or None
#         Path to .vhdr file if found
#     """
#     eeg_path = os.path.join(subject_path, 'ses-1/eeg')
#     print(eeg_path)
#     if not os.path.exists(eeg_path):
#         return None

#     for f in os.listdir(eeg_path):
#         if f.endswith('.vhdr') and task in f:
#             return os.path.join(eeg_path, f)
#     return None

print("EEG loading functions defined successfully!")


EEG loading functions defined successfully!


In [8]:
# =============================================================================
# Cell 6: Feature Extraction Functions
# =============================================================================

# Attempting the canonical import path for psd_welch in MNE 1.11.0
# from mne.time_frequency import psd_welch

def extract_band_powers(raw, freq_bands=FREQ_BANDS):
    """
    Extract power spectral density features for each frequency band.

    Parameters:
    -----------
    raw : mne.io.Raw
        Preprocessed raw EEG data
    freq_bands : dict
        Dictionary of frequency bands {name: (low, high)}

    Returns:
    --------
    features : dict
        Dictionary of band powers for each channel
    """
    # Compute PSD using Welch method
    # psds, freqs = psd_welch(raw, fmin=0.5, fmax=45, n_fft=256, verbose=False)
    spectrum = raw.compute_psd(method="welch", fmin=0.5, fmax=45)
    psds, freqs = spectrum.get_data(return_freqs=True)
    print(len(psds), len(freqs))

    features = {}
    for band_name, (fmin, fmax) in freq_bands.items():
        # Find frequency indices for this band
        freq_mask = (freqs >= fmin) & (freqs <= fmax)
        # Average power in this band for each channel
        band_power = psds[:, freq_mask].mean(axis=1)

        for i, ch_name in enumerate(raw.ch_names):
            features[f"{ch_name}_{band_name}"] = band_power[i]

    return features

def create_epochs_and_features(raw, epoch_duration=10, freq_bands=FREQ_BANDS):
    """
    Split raw data into epochs and extract features from each.

    Parameters:
    -----------
    raw : mne.io.Raw
        Preprocessed raw EEG data
    epoch_duration : int
        Duration of each epoch in seconds
    freq_bands : dict
        Frequency bands for feature extraction

    Returns:
    --------
    epoch_features : list of dicts
        Features for each epoch
    """
    sfreq = raw.info['sfreq']
    data = raw.get_data()
    n_samples = data.shape[1]
    epoch_samples = int(epoch_duration * sfreq)

    epoch_features = []

    for start in range(0, n_samples - epoch_samples, epoch_samples):
        end = start + epoch_samples
        epoch_data = data[:, start:end]

        # Create temporary raw object for this epoch
        epoch_info = mne.create_info(raw.ch_names, sfreq, ch_types='eeg')
        epoch_raw = mne.io.RawArray(epoch_data, epoch_info, verbose=False)

        # Extract band power features
        features = extract_band_powers(epoch_raw, freq_bands)
        epoch_features.append(features)

    return epoch_features

print("Feature extraction functions defined successfully!")

Feature extraction functions defined successfully!


In [12]:
# =============================================================================
# Cell 7: Process All OCD Subjects and Build Feature Dataset
# =============================================================================

def process_subjects(tdbrain_path, subject_ids, label, task='restEC'):
    """
    Process multiple subjects and extract features.

    Parameters:
    -----------
    tdbrain_path : str
        Path to TDBRAIN dataset
    subject_ids : list
        List of subject IDs to process
    label : str
        Label for these subjects (e.g., 'OCD', 'Anxiety', 'Control')
    task : str
        Task name for EEG files

    Returns:
    --------
    all_features : pd.DataFrame
        DataFrame with all extracted features and labels
    """
    all_features = []

    for subj_id in subject_ids:
        print(f"Processing {subj_id}...")
        subject_path = os.path.join(tdbrain_path, subj_id)

        # Find EEG file
        vhdr_file = find_eeg_files(subject_path, task)
        if vhdr_file is None:
            print(f"  No EEG file found for {subj_id}")
            continue

        # Load and preprocess
        raw = load_and_preprocess_eeg(vhdr_file, TARGET_SFREQ, COMMON_CHANNELS)
        if raw is None:
            continue

        # Debugging: Print raw data duration
        print(f"  Raw data duration for {subj_id}: {raw.times[-1]:.2f} seconds")

        # Extract features from epochs
        epoch_features = create_epochs_and_features(raw, EPOCH_DURATION, FREQ_BANDS)

        if not epoch_features:
            print(f"  No epochs created for {subj_id}. Data might be too short for {EPOCH_DURATION}-second epochs.")

        # Add subject info and label to each epoch
        for i, features in enumerate(epoch_features):
            features['subject_id'] = subj_id
            features['epoch'] = i
            features['label'] = label
            all_features.append(features)

        print(f"  Extracted {len(epoch_features)} epochs")

    return pd.DataFrame(all_features)

# Example usage (uncomment when you have the data):
ocd_subject_ids = ocd_subjects['participant_id'].tolist()[:30]  # Limit to 30 subjects
ocd_features = process_subjects(TDBRAIN_PATH, ocd_subject_ids, 'ADHD')
print(f"Total OCD features: {len(ocd_features)}")

print("Subject processing function defined!")
print("Uncomment the example usage above when your data is ready.")

Processing sub-001...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-001/ses-1/eeg
  No EEG file found for sub-001
Processing sub-002...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-002/ses-1/eeg
  No EEG file found for sub-002
Processing sub-003...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-003/ses-1/eeg
  No EEG file found for sub-003
Processing sub-004...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-004/ses-1/eeg
  No EEG file found for sub-004
Processing sub-005...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-005/ses-1/eeg
  No EEG file found for sub-005
Processing sub-006...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-006/ses-1/eeg
  No EEG file found for sub-006
Processing sub-007...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-007/ses-1/eeg
  No EEG file found for sub-007
Processing sub-008...
/content/drive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/sub-008/ses-1/eeg
  No EEG file found for 

In [ ]:
# =============================================================================
# Cell 8: ML Model Training - Classical Models (SVM, Random Forest)
# =============================================================================

def prepare_data_for_ml(features_df):
    """
    Prepare feature DataFrame for ML training.

    Parameters:
    -----------
    features_df : pd.DataFrame
        DataFrame with features and labels

    Returns:
    --------
    X : np.array
        Feature matrix
    y : np.array
        Labels
    feature_names : list
        Names of features
    """
    # Get feature columns (exclude metadata columns)
    meta_cols = ['subject_id', 'epoch', 'label']
    feature_cols = [col for col in features_df.columns if col not in meta_cols]

    X = features_df[feature_cols].values
    y = features_df['label'].values

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    return X, y_encoded, feature_cols, le

def train_classical_models(X, y, cv_folds=5):
    """
    Train and evaluate classical ML models using cross-validation.

    Parameters:
    -----------
    X : np.array
        Feature matrix
    y : np.array
        Labels
    cv_folds : int
        Number of cross-validation folds

    Returns:
    --------
    results : dict
        Dictionary with model performance metrics
    """
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Define models
    models = {
        # 'SVM (RBF)': SVC(kernel='rbf', C=1.0, random_state=42),
        # 'SVM (Linear)': SVC(kernel='linear', C=1.0, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    }

    results = {}
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

    for name, model in models.items():
        print(f"\nTraining {name}...")
        scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
        results[name] = {
            'mean_accuracy': scores.mean(),
            'std_accuracy': scores.std(),
            'scores': scores
        }
        print(f"  Accuracy: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

    return results, scaler

# Example usage (uncomment when you have processed features):
X, y, feature_names, label_encoder = prepare_data_for_ml(ocd_features)
results, scaler = train_classical_models(X, y)

print("Classical ML training functions defined!")

In [ ]:
# =============================================================================
# Cell 9: Deep Learning Model - LSTM
# =============================================================================

def prepare_data_for_lstm(X, y, time_steps=10):
    """
    Reshape data for LSTM input (samples, time_steps, features).
    For feature-based approach, we treat each epoch as a sequence.

    Parameters:
    -----------
    X : np.array
        Feature matrix (samples, features)
    y : np.array
        Labels
    time_steps : int
        Number of time steps for LSTM

    Returns:
    --------
    X_lstm : np.array
        Reshaped data for LSTM
    y_lstm : np.array
        Corresponding labels
    """
    n_samples = X.shape[0]
    n_features = X.shape[1]

    # For simplicity, reshape each sample to (1, n_features) as single time step
    # Or group consecutive epochs from same subject
    X_lstm = X.reshape((n_samples, 1, n_features))

    return X_lstm, y

def build_lstm_model(input_shape, n_classes):
    """
    Build a simple LSTM model for EEG classification.

    Parameters:
    -----------
    input_shape : tuple
        Shape of input data (time_steps, features)
    n_classes : int
        Number of output classes

    Returns:
    --------
    model : keras.Model
        Compiled LSTM model
    """
    model = Sequential([
        LSTM(64, input_shape=input_shape, return_sequences=True),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dense(n_classes, activation='softmax' if n_classes > 2 else 'sigmoid')
    ])

    loss = 'sparse_categorical_crossentropy' if n_classes > 2 else 'binary_crossentropy'
    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])

    return model

def train_lstm_model(X, y, test_size=0.2, epochs=50, batch_size=32):
    """
    Train LSTM model with early stopping.

    Parameters:
    -----------
    X : np.array
        Feature matrix
    y : np.array
        Labels
    test_size : float
        Proportion of data for testing
    epochs : int
        Maximum number of training epochs
    batch_size : int
        Batch size for training

    Returns:
    --------
    model : keras.Model
        Trained model
    history : History
        Training history
    """
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Prepare for LSTM
    X_lstm, y_lstm = prepare_data_for_lstm(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_lstm, y_lstm, test_size=test_size, random_state=42, stratify=y_lstm
    )

    # Build model
    n_classes = len(np.unique(y))
    model = build_lstm_model(X_train.shape[1:], n_classes)

    print(model.summary())

    # Early stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )

    # Evaluate
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {test_acc:.4f}")

    return model, history, scaler, (X_test, y_test)

# Example usage (uncomment when you have features):
model, history, lstm_scaler, test_data = train_lstm_model(X, y)

print("LSTM training functions defined!")

In [ ]:
# =============================================================================
# Cell 10: Visualization and Save Results
# =============================================================================

def plot_training_history(history):
    """
    Plot training and validation accuracy/loss curves.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    # Loss
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'training_history.png'), dpi=150)
    plt.show()

def plot_confusion_matrix(y_true, y_pred, classes):
    """
    Plot confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'confusion_matrix.png'), dpi=150)
    plt.show()

def compare_models(results):
    """
    Compare performance of different models.
    """
    models = list(results.keys())
    accuracies = [results[m]['mean_accuracy'] for m in models]
    stds = [results[m]['std_accuracy'] for m in models]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(models, accuracies, yerr=stds, capsize=5, color='steelblue', alpha=0.8)
    plt.ylabel('Accuracy')
    plt.title('Model Comparison - OCD vs Anxiety Classification')
    plt.ylim(0, 1)

    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'model_comparison.png'), dpi=150)
    plt.show()

def save_results(features_df, results, output_path):
    """
    Save processed features and results to files.
    """
    # Save features
    features_df.to_csv(os.path.join(output_path, 'extracted_features.csv'), index=False)
    print(f"Features saved to {output_path}/extracted_features.csv")

    # Save results summary
    results_summary = pd.DataFrame([
        {'Model': name, 'Mean_Accuracy': r['mean_accuracy'], 'Std_Accuracy': r['std_accuracy']}
        for name, r in results.items()
    ])
    results_summary.to_csv(os.path.join(output_path, 'results_summary.csv'), index=False)
    print(f"Results saved to {output_path}/results_summary.csv")

print("Visualization and saving functions defined!")
print("\n" + "="*70)
print("NOTEBOOK SETUP COMPLETE!")
print("="*70)
print("\nNext steps:")
print("1. Upload your TDBRAIN subset to Google Drive")
print("2. Update TDBRAIN_PATH in Cell 3")
print("3. Run cells in order to process data and train models")
print("4. Adjust parameters (epochs, batch size) based on your compute resources")

# Task
Add a code cell to reinstall the MNE package and then re-run the entire notebook.

## reinstall_mne

### Subtask:
Add a code cell to reinstall the MNE package to fix potential installation issues.


**Reasoning**:
To address the ImportError related to `psd_welch`, reinstalling the `mne` package with `--force-reinstall` can resolve potential corruption or version conflicts. This will ensure a clean installation of the package.



In [ ]:
# Reinstall MNE to fix potential installation issues and ensure latest version compatibility
!pip install mne --force-reinstall -q

print("MNE package reinstalled.")

## rerun_notebook

### Subtask:
Instruct the user to re-run the entire notebook after MNE reinstallation to ensure all changes are applied.


## Summary:

### Data Analysis Key Findings
*   The MNE package was successfully reinstalled using `!pip install mne --force-reinstall -q`.
*   During the reinstallation, several `pip` dependency resolution warnings were noted for other packages like `ipython`, `google-colab`, `tensorflow`, `numba`, and `moviepy`, indicating potential version conflicts; however, these warnings did not prevent the successful reinstallation of `mne`.
*   The reinstallation process concluded with a confirmation message: "MNE package reinstalled."

### Insights or Next Steps
*   The notebook should be re-run in its entirety to ensure that the newly reinstalled MNE package is fully recognized and to verify if any prior `ImportError` issues are resolved.
*   Monitor the notebook's execution after re-running for any new issues that might arise due to the reported dependency conflicts.
